# IS 6482 - Week 7 — Linear Model Selection: Stepwise, Ridge, LASSO

**Author:** Varun Gupta

**Date:** 15 March, 2026

Today’s goal is to understand how to choose among linear models when many predictors are available, and how different methods handle the tradeoff between flexibility and simplicity.

- Start with the full OLS model
- Motivate the need for model selection
- Build models using forward stepwise selection
- Build models using backward stepwise selection
- Introduce LASSO and coefficient shrinkage
- Introduce ridge regression and coefficient shrinkage
- Use 10-Fold cross-validation to tune Ridge and Lasso penalties
- Visualize stability of regression coefficients using bootstrap resampling of train split

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

# For train-test split
from sklearn.model_selection import train_test_split
# So that we can apply different transformations to different columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
# For plain linear regression with p-values
import statsmodels.api as sm
# Ridge for ridge regression, Lasso for Lasso regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error
# For cross validation later
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate, KFold

RANDOM_STATE = 42

The Boston dataset records `medv` (median house value) for 506 neighborhoods around Boston. This dataset has only numeric columns which makes it a commonly used dataset in regression examples. But it has known ethical and contextual limitations. We use it here only as a compact numerical example, not as an endorsement of the variable design or the social assumptions behind the data.

The full data dictionary is:

- `crim` - per capita crime rate by town
- `zn` - proportion of residential land zoned for lots over 25,000 sq.ft.
- `indus` - proportion of non-retail business acres per town.
- `chas` - Charles River dummy variable (1 if tract bounds river; 0 otherwise)
- `nox` - nitric oxides concentration (parts per 10 million)
- `rm` - average number of rooms per dwelling
- `age` - proportion of owner-occupied units built prior to 1940
- `dis` - weighted distances to five Boston employment centres
- `rad` - index of accessibility to radial highways
- `tax` - full-value property-tax rate per \$10,000
- `ptratio` - pupil-teacher ratio by town
- `lstat` - % lower status of the population
- `medv` - Median value of owner-occupied homes in \$1000's

In [ ]:
drive.mount('/content/drive')
Boston = pd.read_csv('/content/drive/MyDrive/Teaching/Data Mining/Boston.csv')
Boston.head()

In [ ]:
# Drop the first column
if 'Unnamed: 0' in Boston.columns:
  Boston = Boston.drop(columns=['Unnamed: 0'])
Boston.info()

## Exploratory Data Analysis
### Heatmap of correlations

In [ ]:
# Heatmap of correlation among all the features
plt.figure(figsize=(8, 6))
# center = 0 makes the neutral color in the color map correspond to 0 correlation
# fmt = '.2f' means the numbers are rounded to 2 decimal places
# annot = True means the correlation numbers are displayed in the heatmap
sns.heatmap(Boston.corr(), center = 0 , fmt = '.2f', annot = True, cmap = 'coolwarm')

### Histogram of columns

In [ ]:
Boston.hist(bins=15, figsize=(10, 10))
plt.show()

### Scatter plot of `medv` against predictors
Including a locally smoothed trend line (lowess)

In [ ]:
# list of columns in the predictors we want
X_cols = Boston.columns.drop('medv').tolist()
num_X = len(X_cols)

# This is just to produce the figures in a 4-by-3 grid
fig, axes = plt.subplots(4, 3, sharey=True, figsize=(10, 7))
axes = axes.flatten() # converting a 2-D array to 1-D

np.seterr(divide='ignore', invalid='ignore') # because of the dummy feature, the lowess returns divide by 0. This is to suppress the error

for i, col in enumerate(X_cols):
    # sns.regplot allows adding thr lowess line (otherwise we can use plain scatter too)
    sns.regplot(x=col, y='medv', data=Boston,
                scatter=True,
                ax = axes[i],
                lowess=True,  # lowess = True produces the locally smoothed trend line
                line_kws={"color": "red"})
    axes[i].grid(True) # Add grid for clarity

plt.tight_layout()
plt.show()


### Reflection
- Are there columns that show skewed behavior?
- How would you like to transform them before further analysis?

## Simple Linear Regression
To begin with let's see which features linear regression identifies as significant. We will standardize the numeric columns first (every predictor except `chas`)

### Variable Transformations and OLS

In [ ]:
y = Boston['medv']
X = Boston.drop(columns=['medv'])

# columns to standardize
num_cols = X.columns.drop('chas').tolist()
# chas is already a dummy encoded column
dummy_cols = ['chas']

# define transformation
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),  # apply StandardScaler to num_cols
        ('cat', 'passthrough', dummy_cols),   # pass chas without transforming
    ],
    remainder="drop",
    # this makes sure the transformed features don't have names that begin with num__ and cat__
    verbose_feature_names_out=False
    )

# fit and apply the transformation
preprocessor.fit(X)
X_proc = preprocessor.transform(X)

# Sometimes the preprocessor loses names by converting to numpy so we are
# converting back X_proc to a Pandas dataframe anr restoring the column names
feature_names = preprocessor.get_feature_names_out().tolist()
X_proc = pd.DataFrame(X_proc, columns=feature_names)

X_proc.head()


In [ ]:
# Add the constant column to get intercept with statsmodel's OLS class
X_sm = sm.add_constant(X_proc)
ols_model = sm.OLS(y, X_sm).fit()
print(ols_model.summary())

### Generating Confidence and Prediction Intervals

In [ ]:
# Generating Prediction intervals
# Here I am asking for the 95% intervals for the
#   the mean : E[medv | features]  -- the 95% interval for this incorporates the uncertainty in the estimated coefficients
#   the observation: medv | features -- the 95% interval for this incorporates the uncertainty in the estimated coefficients AND the unavoidable noise term
predicted_vals = ols_model.get_prediction(X_sm.iloc[0:5])
predicted_vals.summary_frame(alpha=0.05)

## Stepwise Selection Methods
**Note:** For illustration only, in this notebook we are only covering the steps of constructing the potential feature sets. We will not be using cross-validation to do model selection among the potential feature sets

### Custom Forward Stepwise procedure

R comes with a step() function (and stepAIC()) which produces a detailed report of the variables considered at each step for adding/removal. There is no such function in sklearn or statsmodels, so we will use a custom function to replicate the functionality. This creates a wrapper around OLS class of statsmodels.

In [ ]:
# Python does not have a library to do Forward Stepwise selection
# This is a custom function for this purpose (courtesy chatGPT)

def fit_model(X, y, features):
    X_sub = sm.add_constant(X[features], has_constant="add")
    model = sm.OLS(y, X_sub).fit()
    return model

def model_rss(model):
    resid = model.resid
    return float(np.sum(resid**2))

def forward_stepwise_report(X, y, verbose=True, criterion="aic", max_steps=None, full_forward=False):
    """
    Forward stepwise selection with an R-step()-style report.

    Parameters
    ----------
    X : pandas DataFrame
        Candidate predictors.
    y : pandas Series or array
        Response.
    verbose : bool
        Whether to print step-by-step tables.
    criterion : str
        'aic', 'bic', or 'rss'
    max_steps : int or None
        Maximum number of forward steps.
    full_forward : bool
        Whether to perform full forward selection (all variables).

    Returns
    -------
    selected : list
        Selected variables in order.
    history : list of DataFrames
        Per-step candidate tables.
    final_model : statsmodels regression result
    """
    remaining = list(X.columns)
    selected = []
    history = []

    current_model = fit_model(X, y, selected)
    current_rss = model_rss(current_model)
    current_aic = current_model.aic
    current_bic = current_model.bic

    step_num = 0

    while len(remaining) > 0:
        if max_steps is not None and step_num >= max_steps:
            break

        rows = []

        for var in remaining:
            trial_features = selected + [var]
            trial_model = fit_model(X, y, trial_features)
            trial_rss = model_rss(trial_model)

            rows.append({
                "action": f"+ {var}",
                "variable": var,
                "RSS": trial_rss,
                "RSS_improvement": current_rss - trial_rss,
                "AIC": trial_model.aic,
                "BIC": trial_model.bic,
                "Adj_R2": trial_model.rsquared_adj,
            })

        step_table = pd.DataFrame(rows).sort_values(
            by={"aic": "AIC", "bic": "BIC", "rss": "RSS"}[criterion],
            ascending=True
        ).reset_index(drop=True)

        history.append(step_table)

        if verbose:
            print("="*100)
            print(f"\nStep {step_num + 1}")
            print(f"Current model: {selected if selected else ['(intercept only)']}")
            print(step_table.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

        best_row = step_table.iloc[0]
        best_var = best_row["variable"]

        # stopping rule
        if criterion == "aic":
            best_value = best_row["AIC"]
            current_value = current_aic
            improve = best_value < current_value
        elif criterion == "bic":
            best_value = best_row["BIC"]
            current_value = current_bic
            improve = best_value < current_value
        elif criterion == "rss":
            best_value = best_row["RSS"]
            current_value = current_rss
            improve = best_value < current_value
        else:
            raise ValueError("criterion must be 'aic', 'bic', or 'rss'")

        if not improve and not full_forward:
            if verbose:
                print("\nNo further improvement. Stopping.")
            break

        selected.append(best_var)
        remaining.remove(best_var)

        current_model = fit_model(X, y, selected)
        current_rss = model_rss(current_model)
        current_aic = current_model.aic
        current_bic = current_model.bic
        step_num += 1

        if improve:
            best_model = current_model

    return selected, history, best_model

### Forward stepwise report

In [ ]:
selected, history, best_model = forward_stepwise_report(
    X_proc, y, criterion="aic", verbose=True, full_forward=True
)

print("\nSelected variables:", selected)
print(best_model.summary())

### Custom Backward Stepwise procedure

In [ ]:
def backward_stepwise_report(X, y, verbose=True, criterion="aic", min_features=0, full_backward=False):
    """
    Backward stepwise selection with per-step removal table.
    """
    selected = list(X.columns)
    history = []

    current_model = fit_model(X, y, selected)
    current_rss = model_rss(current_model)
    current_aic = current_model.aic
    current_bic = current_model.bic

    step_num = 0

    while len(selected) > min_features:
        rows = []

        for var in selected:
            trial_features = [f for f in selected if f != var]
            trial_model = fit_model(X, y, trial_features)
            trial_rss = model_rss(trial_model)

            rows.append({
                "action": f"- {var}",
                "variable": var,
                "RSS": trial_rss,
                "RSS_change": trial_rss - current_rss,
                "AIC": trial_model.aic,
                "BIC": trial_model.bic,
                "Adj_R2": trial_model.rsquared_adj,
            })

        step_table = pd.DataFrame(rows).sort_values(
            by={"aic": "AIC", "bic": "BIC", "rss": "RSS"}[criterion],
            ascending=True
        ).reset_index(drop=True)

        history.append(step_table)

        if verbose:
            print("="*100)
            print(f"\nStep {step_num + 1}")
            print(f"Current model: {selected}")
            print(step_table.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

        best_row = step_table.iloc[0]
        best_var = best_row["variable"]

        if criterion == "aic":
            best_value = best_row["AIC"]
            current_value = current_aic
            improve = best_value < current_value
        elif criterion == "bic":
            best_value = best_row["BIC"]
            current_value = current_bic
            improve = best_value < current_value
        elif criterion == "rss":
            best_value = best_row["RSS"]
            current_value = current_rss
            improve = best_value < current_value
        else:
            raise ValueError("criterion must be 'aic', 'bic', or 'rss'")

        if not improve and not full_backward:
            if verbose:
                print("\nNo further improvement. Stopping.")
            break

        selected.remove(best_var)
        current_model = fit_model(X, y, selected)
        current_rss = model_rss(current_model)
        current_aic = current_model.aic
        current_bic = current_model.bic
        step_num += 1

        if improve:
            best_model = current_model

    return selected, history, best_model

### Backward Stepwise report

In [ ]:
selected, history, best_model = backward_stepwise_report(
    X_proc, y, criterion="aic", verbose=True, full_backward=True
)

print("\nSelected variables:", selected)
print(best_model.summary())

### Reflection
- Do we get the same subsets in forward vs. backward? What are the differences?
- Why do we see a different selection of variables in the two procedures?

## LASSO selection
Recall the LASSO loss function is
$ Loss = \sum_{i=1}^n (\beta_{0} + \beta_{1}x_{i,1}+\cdots+\beta_{p}x_{i,p} - y_i)^2 + \lambda \sum_{j=1}^p |\beta_j| $

We must normalize the columns to have the same variance so that the coefficient values can be compared and penalized

In [ ]:
# List of penalty parameters to try
lambdas = np.logspace(np.log10(0.001), np.log10(10), 100) # 0.001 to 10

# We will store the coefficients of all the features for all lambdas in this list
coefs = []

# Loop over lambdas
for lam in lambdas:
    # Initialize Lasso regressor by setting alpha = penalty parameter
    lasso = Lasso(alpha=lam, max_iter=100000)
    # Fit using the entire training set
    lasso.fit(X_proc, y)
    # Store the values of the fitted coefficients
    coefs.append(lasso.coef_)

# Convert to a numpy array, and then into a dataframe
coefs = np.array(coefs)
coef_df = pd.DataFrame(coefs, columns=feature_names, index=lambdas)

In [ ]:
# Plot coefficient paths
fig = plt.figure(figsize=(6, 4))

for i, col in enumerate(feature_names):
  plt.plot(
      coef_df.index,
      coef_df[col],
      linewidth=2,
      label=col,
      linestyle="-" if i < 10 else "--"
      )

plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel("Standardized Coefficients")
plt.title("Lasso coefficient paths: Boston housing data")
plt.xlim(lambdas.min(), lambdas.max())
plt.axhline(0, color="black", linewidth=0.7, alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Reflection
- Read the plot starting from large `lambda` to small `lambda`. What is the order in which variables enter the model?
- Is the feature importance (the ranking) same as Forward stepwise? Backward stepwise?
- What is the first place where the ranking differs? Why?

## Ridge regression
Recall the Ridge regression loss function is
$ Loss = \sum_{i=1}^n (\beta_{0} + \beta_{1}x_{i,1}+\cdots+\beta_{p}x_{i,p} - y_i)^2 + \lambda \sum_{j=1}^p (\beta_j)^2 $

We must normalize the columns to have the same variance so that the coefficient values can be compared and penalized

In [ ]:
# List of penalty parameters to try
lambdas = np.logspace(np.log10(0.1), np.log10(100000), 100) # 0.1 to 100000

# We will store the coefficients of all the features for all lambdas in this list
coefs = []

# Loop over lambdas
for lam in lambdas:
    # Initialize Lasso regressor by setting alpha = penalty parameter
    ridge = Ridge(alpha=lam, max_iter=100000)
    # Fit using the entire training set
    ridge.fit(X_proc, y)
    # Store the values of the fitted coefficients
    coefs.append(ridge.coef_)

# Convert to a numpy array, and then into a dataframe
coefs = np.array(coefs)
coef_df = pd.DataFrame(coefs, columns=feature_names, index=lambdas)

In [ ]:
# Plot coefficient paths
fig = plt.figure(figsize=(6, 4))

for i, col in enumerate(feature_names):
  plt.plot(
      coef_df.index,
      coef_df[col],
      linewidth=2,
      label=col,
      linestyle="-" if i < 10 else "--"
      )

plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel("Standardized Coefficients")
plt.title("Ridge coefficient paths: Boston housing data")
plt.xlim(lambdas.min(), lambdas.max())
plt.axhline(0, color="black", linewidth=0.7, alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Using CV for hyperparameter tuning in Ridge and Lasso regression

In the code blocks above, we have only compared the order in which different feature selection methods add the variables. In practice we should use crossvalidation to select the model complexity. In the next code block we will use the RidgeCV and LassoCV classes from sklearn which does all the heavy lifting for Ridge regression.

In [ ]:
# Cross Validation for Ridge Regression

# List of penalty parameters to try
lambdas = np.logspace(np.log10(0.1), np.log10(100000), 50) # 0.1 to 100000

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X_proc, y, test_size=0.2, random_state=RANDOM_STATE)

# Choose cross-validation strategy as 10-fold cross validation
cv_strategy = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# This pipeline will fit and transform the data using our preprocessor defined earlier,
# and then perform cross validation for ridge regression to figure out the best lambda
model = make_pipeline(preprocessor, RidgeCV(alphas=lambdas, cv = cv_strategy, scoring='neg_mean_squared_error'))

# Fit the pipeline
model.fit(X_train, y_train)

# best lambda
print(f'lambda selected by 10-CV {round(float(model[-1].alpha_), 3)}')

# print the coefficients of the final fitted model
print('Coefficients selected by 10-CV')
# model[0] means the first stage
feature_names = model[0].get_feature_names_out()
display(pd.DataFrame(
    [model[-1].coef_] , # Wrap the 1D array in a list to make it a single row in the DataFrame
    columns=feature_names)
)

# Find the cross validation Mean Squared Error
crossval_MSE = -model[-1].best_score_
# Find the test set Mean Squared Error
test_MSE = mean_squared_error(y_test, model.predict(X_test))

print(f'Cross Validation MSE for best model: {round(crossval_MSE, 3)}')
print(f'Test set MSE : {round(test_MSE,3)}')
print(f'Train set R^2 : {round(model.score(X_train, y_train),3 )}')
print(f'Test set R^2 : {round(model.score(X_test, y_test),3 )}')


In [ ]:
# Cross validation for Lasso Regression

# List of penalty parameters to try
lambdas = np.logspace(np.log10(0.001), np.log10(3), 100) # 0.001 to 3
# train-test split
X_train, X_test, y_train, y_test = train_test_split(X_proc, y, test_size=0.2, random_state=RANDOM_STATE)

# Choose cross-validation strategy as 10-fold cross validation
cv_strategy = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# This pipeline will fit and transform the data using our preprocessor defined earlier,
# and then perform cross validation for lasso regression to figure out the best lambda
model = make_pipeline(preprocessor, LassoCV(alphas=lambdas, cv = cv_strategy))

# Fit the pipeline
model.fit(X_train, y_train)

# best lambda
print(f'lambda selected by 10-CV {round(float(model[-1].alpha_), 3)}')

# print the coefficients of the final fitted model
print('Coefficients selected by 10-CV')
feature_names = model[0].get_feature_names_out()
display(pd.DataFrame(
    [model[-1].coef_] , # Wrap the 1D array in a list to make it a single row in the DataFrame
    columns=feature_names)
)

# Need to jump through a few hoops to find the best cross validation MSE
mean_mse_per_alpha = model[-1].mse_path_.mean(axis=1)
# The optimal alpha is stored in model.alpha_
optimal_alpha = model[-1].alpha_
# Find the index of the optimal alpha
optimal_alpha_index = list(model[-1].alphas_).index(optimal_alpha)
# Get the cross-validation MSE at the optimal alpha
crossval_MSE = mean_mse_per_alpha[optimal_alpha_index]

# Find the test set Mean Squared Error
test_MSE = mean_squared_error(y_test, model.predict(X_test))

print(f'Cross Validation MSE for best model: {round(crossval_MSE, 3)}')
print(f'Test set MSE : {round(test_MSE,3)}')
print(f'Train set R^2 : {round(model.score(X_train, y_train),3 )}')
print(f'Test set R^2 : {round(model.score(X_test, y_test),3 )}')


## Coefficient Stability analysis using Bootstrap sampling

Unline statsmodels.OLS, with Ridge regression and LASSO, we do not get p-values so we do not immediately know whether the coefficients we have fitted are significant. One run of RidgeCV gives a single selected penalty and one fitted coefficient vector. There are lots of sources of noise -- the data generating model itself has inherent noise, our dataset is assumed to be a sample from the true population, we perform a random test-train split, we further create K-folds randomly.

To get a sense of how stable these coefficient estimates are across different training samples, in the next code block we wrap RidgeCV inside a bootstrap loop where we will perform the train-test split multiple times, fit a LOOCV Ridge regression model on the train split, and collect the fitted coefficient for each train sample. Since the RidgeCV itself will do Leave-One-Out-CV, there is no additional randomness due to the K-fold cross validation inside it -- the noise comes from the dataset and the random sampling of the training set.The resulting boxplots show the variability of coefficient estimates. (We repeat the same thing for Lasso after this)

In [ ]:
lambdas = np.logspace(np.log10(0.1), np.log10(100000), 50) # 0.1 to 100000

# This pipeline will fit and transform the data using our preprocessor defined earlier,
# and then perform cross validation for ridge regression to figure out the best alpha
# Since we have not passed any cv strategy, RidgeCV uses Leave-One-Out
model = make_pipeline(preprocessor, RidgeCV(alphas=lambdas))

# These two lists store the results of RidgeCV
lambda_list = []
coefs = []

# Note one run of the above model pipeling only produces one set of coefficients (by
# finding the best lambda, and then fitting on the entire training set)
# To get estimates of what the errors in our coefficients might be, we will
# perform the RidgeCV step across multiple random train splits
num_bootstraps = 20
for i in range(num_bootstraps):
    # train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_proc, y, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    lambda_list.append(round(float(model[-1].alpha_), 3))
    coefs.append(model[-1].coef_)

# Extract names of features
feature_names = model[0].get_feature_names_out()
# Convert coefs to a dataframe
coefs = pd.DataFrame(coefs, columns=feature_names)

print("="*80)
print("Lambdas selected across folds: ")
print(lambda_list)
print("Median selected lambda:", np.median(lambda_list))
print("="*80)

# We will have num_bootstrap values for the coefficients for each feature
# We now plot the whisker plot for these
coefs[coefs.median().sort_values().index].plot.box(vert=False)
plt.axvline(x=0, ymin=-1, ymax=1, color="black", linestyle="--")
_ = plt.title(f"Coefficients of Ridge models\n across {num_bootstraps} Train splits")

Predictors whose boxplots remain far from zero tend to have consistently important effects, while predictors with boxplots centered near zero are less stable or less influential under ridge shrinkage.

In [ ]:
lambdas = np.logspace(np.log10(0.001), np.log10(3), 100) # 0.001 to 3

# This pipeline will fit and transform the data using our preprocessor defined earlier,
# and then perform cross validation for lasso regression to figure out the best alpha
# Since we have not passed any cv strategy, LassoCV uses Leave-One-Out
model = make_pipeline(preprocessor, LassoCV(alphas=lambdas))

lambda_list = []
coefs = []

# Note one run of the above model pipeling only produces one set of coefficients (by
# finding the best lambda, and then fitting on the entire training set)
# To get estimates of what the errors in our coefficients might be, we will
# perform the LassoCV step across multiple random train splits
num_bootstraps = 20
for i in range(num_bootstraps):
    # train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_proc, y, test_size=0.2, random_state=i)
    model.fit(X_train, y_train)
    lambda_list.append(round(float(model[-1].alpha_), 3))
    coefs.append(model[-1].coef_)

# Extract names of features
feature_names = model[0].get_feature_names_out()
# Convert coefs to a dataframe
coefs = pd.DataFrame(coefs, columns=feature_names)

print("="*80)
print("Lambdas selected across folds: ")
print(lambda_list)
print("Median selected lambda:", round(np.median(lambda_list),3))
print("="*80)

# We will have num_bootstrap values for the coefficients for each feature
# We now plot the whisker plot for these
coefs[coefs.median().sort_values().index].plot.box(vert=False)
plt.axvline(x=0, ymin=-1, ymax=1, color="black", linestyle="--")
_ = plt.title(f"Coefficients of Lasso models\n across {num_bootstraps} Train splits")